# Análisis de ofertas de Nextgig

Resumen del conjunto de datos, disponibilidad de campos y conteo de ofertas por rol.

In [ ]:
import pandas as pd
df = pd.read_parquet("nextgig_jobs_2026-06.parquet")
print(len(df))
print(df["country"].value_counts().head(10))
cols = ["skills_required","years_experience_numeric","education_level",
        "certifications","languages_required","work_model","date_posted"]
print(df[cols].notna().mean())
print(df["title"].head(10)) #<- en Títulos vienen los roles, hay que generar un script para la busqueda.

# roles = {"data analyst": r"data analyst", "data scientist": r"data scientist",
#          "full stack": r"full[\s-]?stack",
#          "qa": r"\bqa\b|quality assurance|test engineer|sdet",
#          "cloud": r"cloud engineer|devops|site reliability|platform engineer"}
# t = df["title"].str.lower()
# for r, p in roles.items():
#     print(r, t.str.contains(p, regex=True).sum())


112816
country
USA                         35399
United States               20106
US                          13104
United States of America     5480
Canada                       4923
India                        4209
UK                           3147
United Kingdom               2552
France                       2107
Australia                     867
Name: count, dtype: int64
skills_required             0.712771
years_experience_numeric    0.002172
education_level             0.089411
certifications              0.058183
languages_required          0.071612
work_model                  0.772204
date_posted                 0.193483
dtype: float64
0               Implementation Manager
1       Personal Shopper Part Time Day
2                Cashier Part Time Day
3            Cashier Part Time Evening
4          Produce Clerk Part Time Day
5                Cashier Part Time Day
6            Student Part Time Evening
7    Front End Clerk Part Time Evening
8     Student, Franchise Part Tim

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112816 entries, 0 to 112815
Data columns (total 47 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   title                            112816 non-null  str    
 1   normalized_title                 112815 non-null  str    
 2   company_name                     112816 non-null  str    
 3   ats_name                         104263 non-null  str    
 4   industry                         7358 non-null    str    
 5   function                         7682 non-null    str    
 6   occupational_category            2183 non-null    str    
 7   employment_type                  111344 non-null  str    
 8   work_model                       87117 non-null   str    
 9   schedule                         33157 non-null   str    
 10  shift                            13381 non-null   str    
 11  experience_level                 65095 non-null   str    
 12  job_level_nor

In [ ]:
import pandas as pd
sd = pd.read_csv("skill_demand_daily.csv", parse_dates=["date"])

# 1. lo que falta
rango = pd.date_range(sd["date"].min(), sd["date"].max())
print("Faltan:", set(rango) - set(sd["date"]))
print(sd.groupby("date")["tagged"].first().describe())

# 2. skills mas importantes en sector IT
buscadas = ["python","sql","aws","azure","docker","kubernetes","react",
            "javascript","java","tableau","power bi","selenium","excel","ai","machine learning"]
print(sorted(set(buscadas) & set(sd["skill"])))
print(sorted(set(buscadas) - set(sd["skill"])))

# 3. serie semanal y cambio primera vs ultima semana
sd["semana"] = sd["date"].dt.to_period("W")
sem = sd.groupby(["semana","skill"])["share_pct"].mean().unstack("skill")
cambio = pd.DataFrame({
    "inicio": sem.iloc[0], "fin": sem.iloc[-1],
    "dif_pp": sem.iloc[-1] - sem.iloc[0]
}).query("inicio >= 0.5").sort_values("dif_pp")
print(cambio.head(10)); print(cambio.tail(10))

Faltan: {Timestamp('2026-11-07 00:00:00')}
count    3.700000e+01
mean     3.070524e+05
std      3.685574e+05
min      1.414700e+04
25%      1.092260e+05
50%      2.352020e+05
75%      3.931720e+05
max      1.437983e+06
Name: tagged, dtype: float64
['aws', 'azure', 'docker', 'excel', 'java', 'javascript', 'kubernetes', 'python', 'react', 'sql', 'tableau']
['ai', 'machine learning', 'power bi', 'selenium']
            inicio    fin  dif_pp
skill                            
python       35.53  3.055 -32.475
ml           22.99  3.390 -19.600
sql          20.34  1.570 -18.770
aws          18.48  2.000 -16.480
java         12.32  1.620 -10.700
salesforce   11.26  1.280  -9.980
azure        10.48  0.930  -9.550
kubernetes   10.24  0.750  -9.490
typescript   10.17  1.035  -9.135
c++          10.07  0.990  -9.080
             inicio   fin  dif_pp
skill                            
airflow        1.62  0.13   -1.49
swift          1.41  0.14   -1.27
redis          1.29  0.12   -1.17
nosql         

In [4]:
tag = sd.groupby("date")["tagged"].first()
print(tag.head(10)); print(tag.tail(10))

# ¿el ranking relativo es estable aunque cambie el nivel?
rank = sd[sd["date"].isin([sd["date"].min(), sd["date"].max()])] \
        .pivot(index="skill", columns="date", values="share_pct") \
        .rank(ascending=False)
print(rank.sort_values(rank.columns[-1]).head(15))

date
2026-10-04     14147
2026-10-05     22261
2026-10-06     22442
2026-10-07     22318
2026-10-08     30468
2026-10-09     30616
2026-10-10     33020
2026-10-11     34431
2026-10-12    108219
2026-10-13    109226
Name: tagged, dtype: int64
date
2026-10-31     393172
2026-11-01     422314
2026-11-02     451456
2026-11-03     505113
2026-11-04     505113
2026-11-05     505113
2026-11-06     505113
2026-11-08    1375868
2026-11-09    1437666
2026-11-10    1437983
Name: tagged, dtype: int64
date                  2026-10-04  2026-11-10
skill                                       
customer_service             NaN         1.0
talent_acquisition           NaN         2.0
project_management           NaN         3.0
patient_care                 NaN         4.0
ml                           2.0         5.0
python                       1.0         6.0
supply_chain                 NaN         7.0
ms_office                    NaN         8.0
aws                          4.0         9.0
crm        